# FINN - Notebook Miguel - Red cuantizada 16x12 RGB → 2 salidas

Este notebook está adaptado para tu red propia guardada como:

`state_dict_quantized_last.pth`

La red trabaja con imágenes RGB de `16 x 12`, por tanto la entrada es:

`(1, 3, 16, 12)`

Como `3 x 16 x 12 = 576`, la primera capa lineal tiene entrada de 576 valores.

La salida de la red es de tamaño 2, normalmente interpretable como:

- `out0` → velocidad lineal `v`
- `out1` → velocidad angular `w`

In [2]:
# 1. Imports y preparación básica
import os
import sys
import gc
import warnings

warnings.filterwarnings("default")
gc.collect()

print("Python path actual:", sys.path[0])
print("Carpeta actual:", os.getcwd())

Python path actual: /home/miguel_tfg/FINN-v0.9-Stable/src
Carpeta actual: /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example


In [3]:
# 2. Configurar FINN_BUILD_DIR
# FINN necesita esta variable para crear carpetas temporales de generación.

FINN_BUILD_DIR = os.path.abspath("FINN_build")
os.environ["FINN_BUILD_DIR"] = FINN_BUILD_DIR
os.makedirs(FINN_BUILD_DIR, exist_ok=True)

print("FINN_BUILD_DIR =", os.environ["FINN_BUILD_DIR"])

FINN_BUILD_DIR = /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build


In [4]:
# 3. Imports de PyTorch, Brevitas, QONNX y FINN
import torch
import torch.nn as nn

import brevitas.nn as qnn
from brevitas.quant import Int8Bias
import brevitas.onnx as bo

from qonnx.core.modelwrapper import ModelWrapper
from qonnx.transformation.infer_shapes import InferShapes
from qonnx.transformation.fold_constants import FoldConstants
from qonnx.transformation.general import GiveReadableTensorNames, GiveUniqueNodeNames, RemoveStaticGraphInputs

from finn.util.visualization import showInNetron
from finn.util.basic import make_build_dir

print("Torch cargado. CUDA disponible:", torch.cuda.is_available())

/home/miguel_tfg/FINN-v0.9-Stable/deps/brevitas/src/brevitas/__init__.py:5: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  from pkg_resources import get_distribution, DistributionNotFound
/opt/conda/lib/python3.8/site-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('mpl_toolkits')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#keyword-namespace-packages
  declare_namespace(pkg)
/opt/conda/lib/python3.8/site-packages/pkg_resources/__init__.py:3154: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`. See https://setuptools.pypa.io/en/latest/references/keywords.html#ke

Torch cargado. CUDA disponible: False


/opt/conda/lib/python3.8/site-packages/torch/cuda/__init__.py:52: UserWarning: CUDA initialization: Found no NVIDIA driver on your system. Please check that you have an NVIDIA GPU and installed a driver from http://www.nvidia.com/Download/index.aspx (Triggered internally at  /opt/conda/conda-bld/pytorch_1607370172916/work/c10/cuda/CUDAFunctions.cpp:100.)
  return torch._C._cuda_getDeviceCount() > 0


In [5]:
# 4. Parámetros de la red propia

WEIGHTS_FILE = "state_dict_quantized_last.pth"
MODEL_ONNX = "model.onnx"
MODEL_TIDY_ONNX = "model_tidy.onnx"

# MUY IMPORTANTE:
# Tu red NO usa 64x64 en gris.
# Tu red usa 16x12 RGB.
INPUT_SHAPE = (1, 3, 16, 12)

INPUT_FEATURES = 3 * 16 * 12
NUM_OUTPUTS = 2

print("Archivo de pesos:", WEIGHTS_FILE)
print("INPUT_SHAPE:", INPUT_SHAPE)
print("INPUT_FEATURES:", INPUT_FEATURES)
print("NUM_OUTPUTS:", NUM_OUTPUTS)

assert INPUT_FEATURES == 576
assert os.path.exists(WEIGHTS_FILE), f"No encuentro {WEIGHTS_FILE}. Súbelo a la misma carpeta del notebook."

Archivo de pesos: state_dict_quantized_last.pth
INPUT_SHAPE: (1, 3, 16, 12)
INPUT_FEATURES: 576
NUM_OUTPUTS: 2


In [6]:
# 5. Inspección del state_dict
# Esta celda sirve para confirmar que los pesos coinciden con la red esperada.

state_dict_check = torch.load(WEIGHTS_FILE, map_location="cpu")

if isinstance(state_dict_check, dict) and "state_dict" in state_dict_check:
    state_dict_check = state_dict_check["state_dict"]

for k, v in state_dict_check.items():
    if hasattr(v, "shape"):
        print(k, v.shape)
    else:
        print(k, type(v))

quant_input.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fc1.weight torch.Size([256, 576])
fc1.bias torch.Size([256])
relu1.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fc2.weight torch.Size([128, 256])
fc2.bias torch.Size([128])
relu2.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
fx.weight torch.Size([16, 128])
fx.bias torch.Size([16])
relu3.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])
predictor.weight torch.Size([2, 16])
predictor.bias torch.Size([2])
quant_output.act_quant.fused_activation_quant_proxy.tensor_quant.scaling_impl.value torch.Size([])


In [7]:
# 6. Definición de la arquitectura real de tu red
# Estructura detectada desde el state_dict:
# 576 -> 256 -> 128 -> 16 -> 2

class RedMiguel(nn.Module):
    def __init__(self, in_width=8, weight_width=4, act_width=4, out_width=8):
        super(RedMiguel, self).__init__()

        self.quant_input = qnn.QuantIdentity(
            return_quant_tensor=True,
            bit_width=in_width
        )

        self.fc1 = qnn.QuantLinear(
            in_features=576,
            out_features=256,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu1 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.fc2 = qnn.QuantLinear(
            in_features=256,
            out_features=128,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu2 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.fx = qnn.QuantLinear(
            in_features=128,
            out_features=16,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.relu3 = qnn.QuantReLU(
            bit_width=act_width,
            return_quant_tensor=True
        )

        self.predictor = qnn.QuantLinear(
            in_features=16,
            out_features=2,
            weight_bit_width=weight_width,
            bias=True,
            bias_quant=Int8Bias,
            return_quant_tensor=True
        )

        self.quant_output = qnn.QuantIdentity(
            return_quant_tensor=True,
            bit_width=out_width
        )

    def forward(self, x):
        x = self.quant_input(x)

        # Entrada esperada: (batch, 3, 16, 12)
        # Se aplana a 576 valores: RRR... GGG... BBB...
        x = x.reshape(x.size(0), -1)

        x = self.fc1(x)
        x = self.relu1(x)

        x = self.fc2(x)
        x = self.relu2(x)

        x = self.fx(x)
        x = self.relu3(x)

        x = self.predictor(x)
        x = self.quant_output(x)

        return x

print("Arquitectura RedMiguel definida correctamente.")

Arquitectura RedMiguel definida correctamente.


In [8]:
# 7. Crear modelo y cargar los pesos propios

model = RedMiguel(in_width=8, weight_width=4, act_width=4, out_width=8)

state_dict = torch.load(WEIGHTS_FILE, map_location="cpu")

if isinstance(state_dict, dict) and "state_dict" in state_dict:
    state_dict = state_dict["state_dict"]

model.load_state_dict(state_dict)
model.eval()

print("Pesos cargados correctamente desde:", WEIGHTS_FILE)

Pesos cargados correctamente desde: state_dict_quantized_last.pth


In [9]:
# 8. Probar una inferencia dummy en PyTorch
# Si esta celda falla con 4096 vs 576, significa que INPUT_SHAPE sigue mal.

with torch.no_grad():
    dummy_input = torch.randn(*INPUT_SHAPE)
    dummy_output = model(dummy_input)

print("Forma entrada:", tuple(dummy_input.shape))
print("Forma salida:", tuple(dummy_output.shape))
print("Salida dummy:", dummy_output)

Forma entrada: (1, 3, 16, 12)
Forma salida: (1, 2)
Salida dummy: QuantTensor(value=tensor([[-0.1533,  0.1666]]), scale=tensor(0.0067), zero_point=tensor(0.), bit_width=tensor(8.), signed_t=tensor(True), training_t=tensor(False))


In [10]:
# 9. Exportar la red a ONNX compatible con FINN

bo.export_finn_onnx(model, INPUT_SHAPE, MODEL_ONNX)
print("Modelo exportado a:", MODEL_ONNX)

Modelo exportado a: model.onnx


In [11]:
# 10. Comprobar entrada y salida del ONNX

model_check = ModelWrapper(MODEL_ONNX)

print("Entradas:")
for inp in model_check.graph.input:
    print(inp.name, model_check.get_tensor_shape(inp.name))

print("Salidas:")
for out in model_check.graph.output:
    print(out.name, model_check.get_tensor_shape(out.name))

Entradas:
inp.1 [1, 3, 16, 12]
86 [1]
Salidas:
85 [1, 2]


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/core/modelwrapper.py:93: UserWarning: Some old-style domain attributes were automatically converted to new-style,
                i.e. domain=finn to domain=qonnx.custom_op.<general|fpgadataflow|...>
  warnings.warn(


In [12]:
# 11. Limpieza básica del ONNX: nombres, shapes y constantes

model_qonnx = ModelWrapper(MODEL_ONNX)
model_qonnx = model_qonnx.transform(InferShapes())
model_qonnx = model_qonnx.transform(FoldConstants())
model_qonnx = model_qonnx.transform(GiveUniqueNodeNames())
model_qonnx = model_qonnx.transform(GiveReadableTensorNames())
model_qonnx = model_qonnx.transform(RemoveStaticGraphInputs())
model_qonnx.save(MODEL_TIDY_ONNX)

print("Modelo ONNX limpio guardado en:", MODEL_TIDY_ONNX)

Modelo ONNX limpio guardado en: model_tidy.onnx


/home/miguel_tfg/FINN-v0.9-Stable/deps/qonnx/src/qonnx/util/onnx.py:40: DeprecationWarning: `mapping.TENSOR_TYPE_TO_NP_TYPE` is now deprecated and will be removed in the next release or so.To silence this warning, please use `helper.{self._future_function}` instead.
  return np.zeros(dims, dtype=onnx.mapping.TENSOR_TYPE_TO_NP_TYPE[vi.type.tensor_type.elem_type])


In [13]:
# 12. Mostrar estructura del modelo limpio

model_check = ModelWrapper(MODEL_TIDY_ONNX)

print("Entradas:")
for inp in model_check.graph.input:
    print(inp.name, model_check.get_tensor_shape(inp.name))

print("Salidas:")
for out in model_check.graph.output:
    print(out.name, model_check.get_tensor_shape(out.name))

print("Nodos:")
for node in model_check.graph.node:
    print(node.op_type, node.name)

Entradas:
global_in [1, 3, 16, 12]
Salidas:
global_out [1, 2]
Nodos:
MultiThreshold MultiThreshold_0
Add Add_0
Mul Mul_0
Reshape Reshape_0
MatMul MatMul_0
Mul Mul_1
Div Div_0
Add Add_1
Mul Mul_2
MultiThreshold MultiThreshold_1
Mul Mul_3
MatMul MatMul_1
Mul Mul_4
Div Div_1
Add Add_2
Mul Mul_5
MultiThreshold MultiThreshold_2
Mul Mul_6
MatMul MatMul_2
Mul Mul_7
Div Div_2
Add Add_3
Mul Mul_8
MultiThreshold MultiThreshold_3
Mul Mul_9
MatMul MatMul_3
Mul Mul_10
Div Div_3
Add Add_4
Mul Mul_11
MultiThreshold MultiThreshold_4
Add Add_5
Mul Mul_12


In [14]:
# 13. Visualización opcional con Netron
# Descomenta si quieres verlo en Netron.

# showInNetron(MODEL_TIDY_ONNX)

## A partir de aquí empieza el flujo de FINN build_dataflow

Esta parte intenta generar los resultados en `output_final_miguel`.

Si tu versión del notebook original del profesor usaba más pasos concretos, puedes copiar esas celdas debajo de esta sección usando como entrada `model_tidy.onnx` o `model.onnx`, según corresponda.

In [15]:
import os
import shutil

from finn.builder import build_dataflow as build
from finn.builder import build_dataflow_config as build_cfg

final_output_dir = "output_final_miguel"

if os.path.exists(final_output_dir):
    shutil.rmtree(final_output_dir)
    print("Previous run results deleted:", final_output_dir)

cfg = build.DataflowBuildConfig(
    output_dir=final_output_dir,
    generate_outputs=[
        build_cfg.DataflowOutputType.ESTIMATE_REPORTS,
        build_cfg.DataflowOutputType.STITCHED_IP,
    ],
    folding_config_file=None,
    auto_fifo_depths=True,
    mvau_wwidth_max=36,
    target_fps=1000000,
    synth_clk_period_ns=10.0,
    fpga_part="xc7z020clg400-1"
)

print("Configuración creada. Salida:", final_output_dir)

Previous run results deleted: output_final_miguel
Configuración creada. Salida: output_final_miguel


In [16]:
ls -R /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567 | head -100


/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567:
hls_syn_MatrixVectorActivation_0.tcl
ipgen.sh
memblock_sim_0.dat
memblock_synth_0.dat
project_MatrixVectorActivation_0/
thresh.h
top_MatrixVectorActivation_0.cpp
vitis_hls.log
weights.npy

/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567/project_MatrixVectorActivation_0:
hls.app
sol1/

/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567/project_MatrixVectorActivation_0/sol1:
sol1.aps
sol1.directive
sol1.log


In [20]:
!grep -R "ERROR\|Error\|error\|CRITICAL\|failed\|Failed" /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567

/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567/project_MatrixVectorActivation_0/sol1/.autopilot/xcd.log:Fri Jun 12 10:39:02 2026: XCD main fifo: do_read error code: Operation canceled
/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567/project_MatrixVectorActivation_0/sol1/.autopilot/db/autopilot.flow.log:Execute       ap_eval exec -ignorestderr /tools/Xilinx/Vitis_HLS/2022.2/lnx64/tools/clang-3.9-csynth/bin/clang /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567/top_MatrixVectorActivation_0.cpp -foptimization-record-file=/home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build/code_gen_ipgen_MatrixVectorActivation_0_o11cr567/project_MatrixVectorActivation_0/sol1/.autopilot/db/top_MatrixVectorActivation_0.cpp.clang.diag.yml -mllvm -pass-remarks-missed=reflow|pasta|unroll -mllvm -p

In [21]:
import json

folding_config = {
    "Defaults": {},
    "MatrixVectorActivation_0": {
        "PE": 16,
        "SIMD": 8,
        "ram_style": "auto",
        "resType": "auto"
    },
    "MatrixVectorActivation_1": {
        "PE": 16,
        "SIMD": 8,
        "ram_style": "auto",
        "resType": "auto"
    },
    "MatrixVectorActivation_2": {
        "PE": 8,
        "SIMD": 8,
        "ram_style": "auto",
        "resType": "auto"
    },
    "MatrixVectorActivation_3": {
        "PE": 1,
        "SIMD": 8,
        "ram_style": "auto",
        "resType": "auto"
    }
}

with open("folding_config_miguel.json", "w") as f:
    json.dump(folding_config, f, indent=4)

print("Creado folding_config_miguel.json")

Creado folding_config_miguel.json


In [22]:
import os
print(os.path.exists("folding_config_miguel.json"))

True


In [ ]:
# 15. Ejecutar build_dataflow
# Esta celda puede tardar bastante.
# Si falla por configuración de folding, primero pasa el error y se ajusta el folding_config.

build.build_dataflow_cfg(MODEL_TIDY_ONNX, cfg)
print("Build finalizado. Revisa la carpeta:", final_output_dir)

Building dataflow accelerator from model_tidy.onnx
Intermediate outputs will be generated in /home/miguel_tfg/FINN-v0.9-Stable/notebooks/UAH_example/FINN_build
Final outputs will be generated in output_final_miguel
Build log is at output_final_miguel/build_dataflow.log
Running step: step_qonnx_to_finn [1/17]
Running step: step_tidy_up [2/17]
Running step: step_streamline [3/17]
Running step: step_convert_to_hls [4/17]
Running step: step_create_dataflow_partition [5/17]
Running step: step_target_fps_parallelization [6/17]
Running step: step_apply_folding_config [7/17]
Running step: step_generate_estimate_reports [8/17]
Running step: step_hls_codegen [9/17]
Running step: step_hls_ipgen [10/17]


In [ ]:
# 16. Buscar archivos importantes generados

import os

print("Buscando archivos Verilog / Vivado / all_verilog_srcs...")
found = False
for root, dirs, files in os.walk(final_output_dir):
    for f in files:
        if (
            "all_verilog_srcs" in f
            or "finn_design_wrapper" in f
            or f.endswith(".v")
            or f.endswith(".sv")
            or f.endswith(".dcp")
        ):
            print(os.path.join(root, f))
            found = True

if not found:
    print("No se han encontrado aún Verilog/DCP. Puede que el build solo haya generado modelos intermedios/reportes o que falte activar más outputs.")

In [ ]:
# 17. Mostrar árbol resumido de la carpeta de salida

for root, dirs, files in os.walk(final_output_dir):
    level = root.replace(final_output_dir, "").count(os.sep)
    indent = "  " * level
    print(indent + os.path.basename(root) + "/")
    for f in files[:10]:
        print(indent + "  " + f)

## Resumen para el TFG

Con este notebook se hace:

`state_dict_quantized_last.pth` → carga de pesos en PyTorch → exportación a `model.onnx` → limpieza ONNX → preparación para FINN → generación en `output_final_miguel`.

La red usada en este notebook es tu red real:

`576 → 256 → 128 → 16 → 2`

con entrada:

`1 x 3 x 16 x 12`

Esto coincide con el formato de simulación usado en C++:

`RRR... GGG... BBB...`

In [ ]:
import os

for root, dirs, files in os.walk("output_final_miguel"):
    level = root.replace("output_final_miguel", "").count(os.sep)
    indent = "  " * level
    print(indent + os.path.basename(root) + "/")
    for f in files[:15]:
        print(indent + "  " + f)

In [ ]:
import os

for root, dirs, files in os.walk("output_final_miguel"):
    for f in files:
        if (
            "all_verilog_srcs" in f
            or "finn_design_wrapper" in f
            or f.endswith(".v")
            or f.endswith(".sv")
            or f.endswith(".dcp")
            or f.endswith(".xo")
        ):
            print(os.path.join(root, f))